# 1-2. 토큰화 심화

⏱ **소요시간**: 45분

## 학습 목표
- BPE / WordPiece / SentencePiece 알고리즘의 차이를 설명할 수 있다.
- `tiktoken`으로 동일 의미의 영·한 문장의 토큰 수 차이를 측정한다.
- Llama3 / Qwen2.5 / bert-multilingual 토크나이저의 한국어 효율을 벤치마크한다.
- 토큰 효율 차이가 실제 API 비용에 주는 영향을 계산한다.

## 핵심 키워드
`BPE`, `WordPiece`, `SentencePiece`, `어휘 크기`, `OOV`, `subword`, `토큰 연비(fertility)`

## 0. 왜 토큰화가 중요한가?

LLM은 **토큰 단위**로 입출력을 수행한다. 둘은 용어로 말하면:

- **비용**: OpenAI · Anthropic API는 토큰당 과금한다.
- **속도**: 같은 의미를 적은 토큰으로 표현하는 모델이 빠르다.
- **컨텍스트 윈도우**: 같은 4K 윈도우라도 토큰 효율이 나쁜 모델은 실제 데이터를 테 담는다.
- **한국어 간결 문제**: 영어 중심 토크나이저는 한국어를 자주 **자소 단위**로 깨무린다.

이번 노트북에서는 이 차이를 실제로 눈으로 확인한다.

## 1. 토큰화 알고리즘 배경

| 알고리즘 | 핵심 아이디어 | 대표 모델 |
|---|---|---|
| **BPE** (Byte-Pair Encoding) | 가장 자주 등장하는 문자 쌍을 병합 | GPT-2, GPT-3, GPT-4, Llama |
| **WordPiece** | likelihood 가 증가하는 쌍을 병합, 접두어·추가 subword는 `##` 접두사 | BERT, DistilBERT |
| **SentencePiece** (BPE/Unigram) | 공백을 특수 문자 `▁`로 대체해 언어 독립적, 디토크나이제이션 될돌림 | T5, ALBERT, XLM-R, Llama2, Gemma |
| **Tiktoken BPE (cl100k/o200k)** | BPE를 byte 레벨에 적용 (UTF-8 raw byte) → OOV 없음 | GPT-3.5/4/4o |

### BPE 의사코드
```text
1. 모든 단어를 문자 단위로 분해한다.            ("low </w>") → l o w </w>
2. 인접 쌍을 세고, 빈도가 가장 높은 쌍을 병합.    (l, o) → "lo"
3. 어휘 크기(V)에 도달할 때까지 반복.
```

### WordPiece 의사코드
```text
1. BPE처럼 문자 단위로 시작.
2. 모든 후보 쌍 (a, b)에 대해 score(ab) = count(ab) / (count(a)*count(b))
3. score 가 가장 큰 쌍을 병합.
```

In [ ]:
import sys
sys.path.insert(0, '../..')

# 준비
import tiktoken
from transformers import AutoTokenizer

print('tiktoken OK')
print('transformers OK')

## 2. tiktoken으로 영/한 토큰 수 비교

`o200k_base`는 GPT-4o 계열, `cl100k_base`는 GPT-4 / GPT-3.5-turbo 계열의 인코딩이다.

In [ ]:
enc_o200k = tiktoken.get_encoding('o200k_base')  # gpt-4o
enc_cl100k = tiktoken.get_encoding('cl100k_base')  # gpt-4 / 3.5-turbo

# 동일한 의미의 영·한 문장 쌍 (금융·행정 주제)
pairs = [
    ('The customer submitted an application for an electronic banking service.',
     '고객이 전자금융 서비스 신청서를 제출했습니다.'),
    ('Banks must protect personal information under the Personal Information Protection Act.',
     '은행은 개인정보보호법에 따라 개인정보를 보호해야 합니다.'),
    ('Network separation is a core principle of financial security.',
     '망분리는 금융 보안의 핵심 원칙입니다.'),
    ('Large language models are based on the Transformer architecture.',
     '대규모 언어모델은 트랜스포머 구조를 기반으로 합니다.'),
    ('Retrieval-Augmented Generation reduces hallucination by grounding answers in documents.',
     '검색 증강 생성은 답변을 문서에 근거해 환각을 줄입니다.'),
    ('The airgapped environment physically isolates internal networks from the Internet.',
     '폐쇄망 환경은 내부망을 인터넷과 물리적으로 분리합니다.'),
    ('Knowledge graphs represent entities and relations as nodes and edges.',
     '지식 그래프는 개체와 관계를 노드와 엣지로 표현합니다.'),
]

print(f'{"EN tok":>8} {"KO tok":>8} {"ratio":>8}  text')
print('-' * 70)
total_en, total_ko = 0, 0
for en, ko in pairs:
    en_tok = len(enc_o200k.encode(en))
    ko_tok = len(enc_o200k.encode(ko))
    total_en += en_tok
    total_ko += ko_tok
    print(f'{en_tok:>8} {ko_tok:>8} {ko_tok / en_tok:>8.2f}x  {ko[:30]}...')

print('-' * 70)
print(f'total   EN={total_en}  KO={total_ko}  ratio={total_ko / total_en:.2f}x')

→ 동일 의미라도 GPT-4o 기준 한국어가 영어 대비 1.3~1.8배의 토큰을 사용한다 (o200k 기준). 초기 cl100k에서는 2.5배 이상이었지만, o200k는 한국어 merges를 더 많이 포함해 개선됨.

In [ ]:
# cl100k 와 o200k 직접 비교 (동일 한국어 문장)
ko_texts = [p[1] for p in pairs]
cl100k_total = sum(len(enc_cl100k.encode(t)) for t in ko_texts)
o200k_total = sum(len(enc_o200k.encode(t)) for t in ko_texts)
print(f'cl100k (GPT-4 / 3.5-turbo) : {cl100k_total} tokens')
print(f'o200k  (GPT-4o)           : {o200k_total} tokens')
print(f'개선율: {(1 - o200k_total / cl100k_total) * 100:.1f}% 줄어듦')

## 3. 토큰화 시각적 확인

한국어는 실제로 어떻게 쌊녀지 보자.

In [ ]:
def show_tokens(encoder, text, label):
    ids = encoder.encode(text)
    pieces = [encoder.decode([i]) for i in ids]
    print(f'[{label}] {len(ids)} tokens')
    print('  →', ' | '.join(pieces))


sentence_ko = '전자금융거래법은 금융소비자를 보호한다.'
show_tokens(enc_o200k, sentence_ko, 'o200k (gpt-4o)')
show_tokens(enc_cl100k, sentence_ko, 'cl100k (gpt-4)')

sentence_en = 'The Electronic Financial Transactions Act protects financial consumers.'
show_tokens(enc_o200k, sentence_en, 'o200k (gpt-4o) EN')

## 4. 오픈소스 토크나이저 한국어 효율 벤치마크

폐쇄망에 내제하려는 모델 후보들에 대해 한국어 토큰 효율을 측정한다.

지표:
- **토큰/문자 비율** (작을수록 좋음)
- **문자/토큰 비율** (fertility 역수, 클수록 좋음)

In [ ]:
# 샘플 코퍼스 로드
from pathlib import Path
corpus = Path('../../data/corpus_ko.txt').read_text(encoding='utf-8')
print('전체 문자 수:', len(corpus))
print('아래는 첫 200자:\n', corpus[:200])

In [ ]:
# 모델별 토크나이저 로드 (실행 시 최초에 다운로드 발생)
# 폐쇄망 에서는 사전 다운로드된 모델을 사용.
TOKENIZERS_TO_COMPARE = {
    'bert-base-multilingual-cased': 'bert-base-multilingual-cased',
    'Llama-3.1-8B (proxy)': 'NousResearch/Meta-Llama-3.1-8B',  # gated이라 대체 레포
    'Qwen2.5-7B': 'Qwen/Qwen2.5-7B',
    'xlm-roberta-base': 'xlm-roberta-base',
}

loaded = {}
for label, model_id in TOKENIZERS_TO_COMPARE.items():
    try:
        loaded[label] = AutoTokenizer.from_pretrained(model_id)
        print(f'OK  {label:<35}  vocab={loaded[label].vocab_size}')
    except Exception as e:
        print(f'SKIP {label:<35}  ({type(e).__name__})')

In [ ]:
# 벤치마크
results = []
n_chars = len(corpus)

# tiktoken 도 비교에 포함
for name, enc in [('tiktoken-o200k (gpt-4o)', enc_o200k), ('tiktoken-cl100k (gpt-4)', enc_cl100k)]:
    n_tokens = len(enc.encode(corpus))
    results.append((name, n_tokens, n_chars / n_tokens))

for label, tok in loaded.items():
    n_tokens = len(tok.encode(corpus, add_special_tokens=False))
    results.append((label, n_tokens, n_chars / n_tokens))

print(f'{"tokenizer":<40} {"tokens":>8} {"chars/tok":>12}')
print('-' * 64)
for name, n_tok, ratio in sorted(results, key=lambda r: -r[2]):
    print(f'{name:<40} {n_tok:>8} {ratio:>12.3f}')

print()
print('* chars/tok 가 클수록 한국어 효율이 높음')

해석 가이드

- BERT multilingual: 어휘가 11만이라 한국어 자소가 많아 효율이 낮다.
- Llama3 / Qwen2.5: 어휘 12만~15만 수준 + 다국어 데이터 학습 → cl100k 대비 한국어가 효율적.
- tiktoken o200k: GPT-4o 용. 한국어 merges를 대폭 추가해 cl100k 대비 30~40% 개선.
- xlm-roberta: 원래 다국어 용으로 학습되어 한국어 어휘가 많은 편.

## 5. 실제 비용 계산

gpt-4o-mini 요금 (2024-11 기준):
- Input: $0.150 / 1M tokens
- Output: $0.600 / 1M tokens

전자금융감독규정 전체 본문(약 15만자)을 context에 넣는 시나리오.

In [ ]:
CORPUS_CHARS = 150_000  # 규정 전체 문자 수 (가정)
OUTPUT_TOKENS = 500     # 요약 답변

# 위에서 계산된 chars/tok 비율을 사용
tok_ratios = {name: ratio for name, _, ratio in results}

PRICING_PER_M = {'gpt-4o-mini': (0.150, 0.600)}  # (input, output)

input_per_m, output_per_m = PRICING_PER_M['gpt-4o-mini']
print(f'{"tokenizer":<40} {"input tok":>10} {"$/req":>10}')
print('-' * 64)
for name in ['tiktoken-o200k (gpt-4o)', 'tiktoken-cl100k (gpt-4)']:
    if name not in tok_ratios:
        continue
    chars_per_tok = tok_ratios[name]
    input_tokens = int(CORPUS_CHARS / chars_per_tok)
    cost = (input_tokens * input_per_m + OUTPUT_TOKENS * output_per_m) / 1_000_000
    print(f'{name:<40} {input_tokens:>10} ${cost:>9.4f}')

print()
print('시나리오: 1만 빌 시행하면 o200k 기준 전체 비용?')
chars_per_tok = tok_ratios['tiktoken-o200k (gpt-4o)']
input_tokens = int(CORPUS_CHARS / chars_per_tok)
batch_cost = 10_000 * (input_tokens * input_per_m + OUTPUT_TOKENS * output_per_m) / 1_000_000
print(f'=> ${batch_cost:,.2f}')

<!-- optional -->

## 6. 실전 팁: 한국어 픈러스 절약 기술

1. **불필요한 공백·줄바꿈 제거** — PDF에서 추출한 텍스트는 공백이 매우 많을 수 있다.
2. **긴 숫자 · 테이블 → markdown 변환** — BPE는 숫자를 자릿단위로 쌊는 경향이 있다.
3. **프롬프트 영어 컨버전** — system prompt를 영어로 쓰면 지시부 토큰 40~60% 절감 가능. 단, 한국어 답변 요구는 명시.
4. **cached input 활용** — OpenAI 기준 cached prompt는 입력 비용 50% 할인.

In [ ]:
# <!-- optional -->
# 영어 system prompt 대 한국어 system prompt 토큰 수 비교
sys_ko = ('당신은 한국어 금융 규제 전문가입니다. 정확하고 간결하게 답변하고, '
          '법령 조문 번호를 반드시 인용하며, 불확실한 정보는 추측하지 않습니다.')
sys_en = ('You are a Korean financial regulation expert. Answer accurately and concisely, '
          'cite statute numbers, and do not speculate on uncertain facts.')
print(f'KO system prompt: {len(enc_o200k.encode(sys_ko))} tokens')
print(f'EN system prompt: {len(enc_o200k.encode(sys_en))} tokens')

## 과제

1. 사내 사용 상 충분히 면밀하지 않은 현업 문서 3개(계약서, 정책, FAQ 등)를 준비해, 본 노트북의 4절 벤치마크를 재현시키세요. 어떤 모델이 가장 한국어 효율이 높은지 확인해 보세요.
2. 동일한 질의에 대해 프롬프트를 한국어 → 영어로 바꾸고, 답변 품질이 유지되는지 비교하세요.
3. `tiktoken`의 `encoding_for_model('gpt-4o-mini')`와 `get_encoding('o200k_base')`의 결과가 같은지 확인하세요.

## 더 읽어보기
- Sennrich et al., [Neural Machine Translation of Rare Words with Subword Units (BPE, 2016)](https://arxiv.org/abs/1508.07909)
- Kudo & Richardson, [SentencePiece: A simple and language independent subword tokenizer (2018)](https://arxiv.org/abs/1808.06226)
- [OpenAI Tokenizer Playground](https://platform.openai.com/tokenizer)
- [langchain-kr · 04-Model — LangChain 모델 사용법](https://github.com/teddylee777/langchain-kr/tree/main/04-Model)